# Create Meeting Minutes from an Audio File

The goal of this project is to use an audio file to generate structured meeting minutes, including summaries, discussion points, takeaways, and action items.

For this project, we will:

1. Transcribe the meeting audio
2. Analyze the transcript using Llama
3. Generate structured meeting minutes
4. Build a Gradio interface to make the application interactive

You can use an existing meeting audio file or record something of your own.

## Pro-tip for using Colab

This project requires a GPU to run the models.

If Colab disconnects your GPU:

1. Disconnect and delete the runtime
2. Reload the Colab
3. Connect to a new T4 GPU
4. Confirm that the GPU is available
5. Run the notebook again from the top

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6 gradio

In [ ]:
# imports

import os
import requests
import gradio as gr

from threading import Thread
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    TextIteratorStreamer,
    BitsAndBytesConfig
)

import torch

In [ ]:
# Connect this Colab to your Google Drive

drive.mount("/content/drive")

audio_filename = "/content/denver_extract.mp3"

# Audio File

You can use the Denver City Council meeting audio or use your own meeting audio file.

The audio will first be transcribed into text and then analyzed by Llama.

In [ ]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_Token')
login(hf_token, add_to_git_credential=True)

# Open the file

audio_file = open(audio_filename, "rb")

# STEP 1: Transcribe Audio

## Option 1: Use Open Source for Transcription - Hugging Face Pipelines

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)

transcription = result["text"]

print(transcription)

In [ ]:
open_source_transcription = transcription

## Option 2: Use OpenAI for Transcription

In [ ]:
# Sign in to OpenAI using Secrets in Colab

AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
print(transcription)

In [ ]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

# STEP 2: Analyze & Report

In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


## Configure Llama

We use 4-bit quantization to reduce the GPU memory required to run the Llama model.

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

## Generate Meeting Minutes

We now load Llama, provide the meeting transcript, and generate the meeting minutes.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

In [ ]:
response = tokenizer.decode(outputs[0])

In [ ]:
display(Markdown(response))

# Student contribution

Student Emad S. has made this powerful variation that uses `TextIteratorStreamer` to stream back results into a Gradio UI, and takes advantage of background threads for performance! I'm sharing it here if you'd like to take a look at some very interesting work. Thank you, Emad!

https://colab.research.google.com/drive/1Ja5zyniyJo5y8s1LKeCTSkB2xyDPOt6D

# STEP 3: Create a Gradio Interface

The meeting-minutes pipeline is now working.

Next, we will create a simple Gradio application where a user can upload an audio file and automatically generate meeting minutes.

For the application:

1. OpenAI transcribes the uploaded audio.
2. Llama generates the meeting minutes.
3. `TextIteratorStreamer` streams the generated text.
4. Gradio displays the transcript and meeting minutes.

## Create the Transcription Function

First, we convert our existing OpenAI transcription code into a function.

This allows Gradio to send any uploaded audio file to OpenAI for transcription.

In [ ]:
def transcribe_audio(audio_filename):

    with open(audio_filename, "rb") as audio_file:

        transcription = openai.audio.transcriptions.create(
            model=AUDIO_MODEL,
            file=audio_file,
            response_format="text"
        )

    return transcription

## Create the Meeting Minutes Function

Next, we convert our existing Llama generation code into a function.

Instead of `TextStreamer`, we use `TextIteratorStreamer` so that the generated meeting minutes can be sent progressively to Gradio.

In [ ]:
def generate_minutes(transcription):

    user_prompt = f"""
Below is a meeting transcript.

Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt"
    ).to("cuda")

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generation_kwargs = {
        "input_ids": inputs,
        "max_new_tokens": 2000,
        "streamer": streamer
    }

    thread = Thread(
        target=model.generate,
        kwargs=generation_kwargs
    )

    thread.start()

    generated_text = ""

    for text in streamer:
        generated_text += text
        yield generated_text

### TextStreamer vs TextIteratorStreamer

Earlier, we used `TextStreamer` to display Llama's generated text directly in the notebook.

For the Gradio application, we use `TextIteratorStreamer`.

`TextIteratorStreamer` allows us to receive the generated text piece by piece and send each update to the user interface.

A background `Thread` allows Llama to continue generating while Gradio receives the streamed text.

## Connect the Complete Pipeline

We now connect audio transcription and meeting-minute generation.

The uploaded audio is first transcribed using OpenAI. The transcript is then passed to Llama to generate the meeting minutes.

In [ ]:
def process_meeting(audio_filename):

    if audio_filename is None:
        yield "", "Please upload an audio file."
        return

    transcription = transcribe_audio(audio_filename)

    for minutes in generate_minutes(transcription):
        yield transcription, minutes

# STEP 4: Build the Gradio App

Finally, we create the user interface.

The user can upload a meeting audio file and the application will display:

- the meeting transcript
- the generated meeting minutes

In [ ]:
demo = gr.Interface(
    fn=process_meeting,

    inputs=gr.Audio(
        type="filepath",
        label="Upload Meeting Audio"
    ),

    outputs=[
        gr.Textbox(
            label="Meeting Transcript",
            lines=10
        ),

        gr.Markdown(
            label="Meeting Minutes"
        )
    ],

    title="AI Meeting Minutes Generator",

    description=(
        "Upload a meeting audio file to generate "
        "a transcript and structured meeting minutes."
    )
)

demo.launch()